In [ ]:
def bar_plot(col_alvo = "loan_intent", d1 = "loan_amnt", viz_d1 = "count", d2 = None, viz_d2 = None, w = "loan_amnt"):
  #col_alvo (x) 
  #d1 (y1)
  #viz_y1 (y1): count, volume, media, media_pond
  #d2 (y2)
  #viz_y2 (y2): count, volume, media, media_pond
  
  #Configura d1
  if viz_d1 == "count":
    viz_d1 = (
      df.groupby(col_alvo)[d1]
      .count().sort_index()
    )
  elif viz_d1 == "volume":  
    viz_d1 = (
      df.groupby(col_alvo)[d1]
      .sum().sort_index()
    )
  elif viz_d1 == "media":  
    viz_d1 = (
      df.groupby(col_alvo)[d1]
      .mean().sort_index()
    )
  elif viz_d1 == "media_pond": 
    viz_d1 = (
      df.groupby(col_alvo)
        .apply(lambda g: (g[w] * g[d1]).sum() / g[w].sum())
    )

  #Configura d2
  if viz_d2 == "count":
    viz_d2 = (
      df.groupby(col_alvo)[d2]
      .count().sort_index()
    )
  elif viz_d2 == "volume":  
    viz_d2 = (
      df.groupby(col_alvo)[d2]
      .sum().sort_index()
    )
  elif viz_d2 == "media":  
    viz_d2 = (
      df.groupby(col_alvo)[d2]
      .mean().sort_index()
    )
  elif viz_d2 == "media_pond": 
    viz_d2 = (
      df.groupby(col_alvo)
        .apply(lambda g: (g[w] * g[d2]).sum() / g[w].sum())
    )

  resumo = viz_d1.to_frame().merge(viz_d2.to_frame(), left_index=True, right_index=True)


  fig, ax1 = plt.subplots(figsize=(10, 5))
  ax1.bar(viz_d1.index, viz_d1.values, color="steelblue", alpha=0.7)
  ax1.set_ylabel(d1)

  ax2 = ax1.twinx()
  ax2.plot(viz_d2.index, viz_d2.values, color="orange", marker="o")
  ax2.set_ylabel(d2)

  print(1)
  return fig, print(resumo)

In [ ]:
#A ajustar
def bar_plot_qtde_ly(col_alvo="loan_intent", col_media=None, ponderada=False, peso=None):
    qtde = df[col_alvo].value_counts().sort_index()

    if col_media is None:
        fig = go.Figure()
        fig.add_bar(
            x=qtde.index,
            y=qtde.values,
            name="Quantidade",
            marker_color="steelblue",
            opacity=0.7
        )
        fig.update_layout(
            title=f"Quantidade por {col_alvo}",
            xaxis_title=col_alvo,
            yaxis_title="Quantidade",
            template="plotly_white"
        )
        fig.show()
        return

    if ponderada and peso is not None:
        taxa = (
            df.groupby(col_alvo)
              .apply(lambda g: (g[peso] * g[col_media]).sum() / g[peso].sum())
              .reindex(qtde.index)
        )
        nome_taxa = f"{col_media} ponderada"
    else:
        taxa = (
            df.groupby(col_alvo)[col_media]
              .mean()
              .sort_index()
              .reindex(qtde.index)
        )
        nome_taxa = f"{col_media} média"

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_bar(
        x=qtde.index,
        y=qtde.values,
        name="Quantidade",
        marker_color="steelblue",
        opacity=0.7,
        secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=taxa.index,
            y=taxa.values,
            mode="lines+markers",
            name=nome_taxa,
            line=dict(color="orange", width=3),
            marker=dict(size=8)
        ),
        secondary_y=True
    )

    fig.update_layout(
        title=f"Quantidade e {nome_taxa} por {col_alvo}",
        xaxis_title=col_alvo,
        yaxis_title="Quantidade",
        yaxis2_title=nome_taxa,
        template="plotly_white",
        legend=dict(orientation="v", x=1.02, y=1),
        width=1000,
        height=600
    )

    fig.show()